# 04 - Modelação de Aprendizagem Automática e Planeamento Espaço-Temporal A*
**Unidades Curriculares:** Aprendizagem Automática (AAUTIA) & Sistemas de Decisão e Planeamento (PLNTDIA)

Este notebook demonstra a integração completa da arquitetura **Prever para Planear** (*Predict to Plan*):
1. **Modelos Físicos Baselines:** Constant Velocity (CV) e Constant Acceleration (CA).
2. **Modelos de Regressão ML:** Decision Tree, Random Forest e HistGradientBoosting para predição contínua de $(\Delta x, \Delta y)$ a 1s, 3s e 5s.
3. **Classificação de Manobras:** Previsão da intenção do veículo com Random Forest e HistGradientBoosting.
4. **Planeador A* Espaço-Temporal:** Navegação longitudinal suave e evasão de múltiplos obstáculos dinâmicos em malha fechada.


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.features.pandas_feature_pipeline import PandasFeaturePipeline
from src.models.trajectory_predictor import (
    ConstantVelocityPredictor,
    ConstantAccelerationPredictor,
    MLTrajectoryPredictor,
    ManeuverClassifier,
    evaluate_trajectory_predictor,
    evaluate_maneuver_classifier,
)
from src.planning.astar_planner import AStarTrajectoryPlanner


## 1. Carregamento e Partição dos Dados


In [ ]:
df = pd.read_csv(ROOT / "data" / "processed" / "predict_to_plan_samples.csv")
df_train, df_test = PandasFeaturePipeline.create_scenario_based_split(df, test_size=0.25, random_state=42)
print(f"Treino: {len(df_train)} amostras | Teste: {len(df_test)} amostras")


## 2. Treino e Avaliação dos Modelos de Predição de Trajetória


In [ ]:
predictors = [
    ConstantVelocityPredictor(),
    ConstantAccelerationPredictor(),
    MLTrajectoryPredictor(model_type="decision_tree", max_depth=6),
    MLTrajectoryPredictor(model_type="random_forest", n_estimators=60, max_depth=8),
    MLTrajectoryPredictor(model_type="gradient_boosting", max_iter=80, max_depth=5),
]

results = []
trained_models = {}
for p in predictors:
    if isinstance(p, MLTrajectoryPredictor):
        p.fit(df_train)
        trained_models[p.model_type] = p
    m, _ = evaluate_trajectory_predictor(p, df_test)
    results.append(m.__dict__)

df_metrics = pd.DataFrame(results)
df_metrics


## 3. Treino e Avaliação do Classificador de Manobras / Intenções


In [ ]:
classifiers = [
    ManeuverClassifier(model_type="random_forest", n_estimators=60, max_depth=8),
    ManeuverClassifier(model_type="gradient_boosting", max_iter=80, max_depth=5),
]

clf_results = []
for c in classifiers:
    c.fit(df_train)
    res = evaluate_maneuver_classifier(c, df_test)
    clf_results.append({
        "Modelo": c.name,
        "Accuracy": res["accuracy"],
        "F1-Macro": res["f1_macro"],
        "F1-Weighted": res["f1_weighted"]
    })

pd.DataFrame(clf_results)


## 4. Importância das Características (Random Forest)


In [ ]:
rf_model = trained_models["random_forest"]
importances = rf_model.get_feature_importances().head(10)

plt.figure(figsize=(8, 4))
importances[::-1].plot(kind="barh", color="#2563eb")
plt.title("Top-10 Features Mais Preditivas (Random Forest)")
plt.xlabel("Gini Importance")
plt.grid(axis="x", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()


## 5. Simulação do Planeador A* Espaço-Temporal com Evasão de Obstáculos


In [ ]:
# Configuração do planeador A*
planner = AStarTrajectoryPlanner(horizon_steps=30, dt=0.1, max_speed=18.0, safety_radius=3.5)

# Testar um cenário interativo com aproximação de obstáculo
scene_sample = df_test[(df_test["distance_to_ego"] < 25.0) & (df_test["rel_x"] > 2.0)].iloc[0]

ego_v = float(scene_sample["ego_speed"])
s_pts = np.linspace(0, 60.0, 60)
ego_path = np.column_stack([
    scene_sample["ego_x"] + s_pts * np.cos(scene_sample["ego_heading"]),
    scene_sample["ego_y"] + s_pts * np.sin(scene_sample["ego_heading"]),
])

# Trajetória prevista pelo modelo ML
rf_p = rf_model.predict_displacements(pd.DataFrame([scene_sample]))
p_dx = float(rf_p["3s"][0, 0])
p_dy = float(rf_p["3s"][0, 1])
pred_traj = [(scene_sample["agent_x"] + p_dx * (step / 30.0),
              scene_sample["agent_y"] + p_dy * (step / 30.0)) for step in range(31)]

s_tr, v_tr, a_tr = planner.plan_trajectory(
    ego_init_v=ego_v,
    ego_path_xy=ego_path,
    predicted_obstacles={int(scene_sample["track_id"]): pred_traj}
)

time_axis = np.linspace(0, 3.0, len(s_tr))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(time_axis, s_tr, color="#10b981", linewidth=2)
axes[0].set_title("Perfil de Distância s(t)")
axes[0].set_xlabel("Tempo (s)")
axes[0].set_ylabel("Progresso Longitudinal (m)")
axes[0].grid(True, linestyle="--", alpha=0.5)

axes[1].plot(time_axis, v_tr, color="#3b82f6", linewidth=2)
axes[1].set_title("Perfil de Velocidade v(t)")
axes[1].set_xlabel("Tempo (s)")
axes[1].set_ylabel("Velocidade (m/s)")
axes[1].grid(True, linestyle="--", alpha=0.5)

axes[2].plot(time_axis, a_tr, color="#f59e0b", linewidth=2)
axes[2].set_title("Perfil de Aceleração a(t) [Conforto]")
axes[2].set_xlabel("Tempo (s)")
axes[2].set_ylabel("Aceleração (m/s²)")
axes[2].grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

print(f"Planeamento concluído: Distância final percorrida = {s_tr[-1]:.2f}m em 3.0 segundos.")
